In [1]:
#%%
import torch
import onnxruntime as ort
import os
import numpy as np
import onnx
import time
import itertools

# ===============================
# CONFIG
# ===============================
lat_perturb = True  # ✅ 위도별 표준편차(=axis=3 기반 perturbation) 활성화 옵션
pangu_dir = r'/home1/jek/Pangu-Weather'

lat_indices = np.linspace(90, -90, 721)
lon_indices = np.linspace(-180, 180, 1441)[:-1]

def latlon_extent(lon_min, lon_max, lat_min, lat_max):    
    lon_min, lon_max = lon_min-180, lon_max-180  
    lat_start = np.argmin(np.abs(lat_indices - lat_max)) 
    lat_end = np.argmin(np.abs(lat_indices - lat_min))
    lon_start = np.argmin(np.abs(lon_indices - lon_min))
    lon_end = np.argmin(np.abs(lon_indices - lon_max))
    latlon_ratio = (lon_max-lon_min)/(lat_max-lat_min)
    extent=[lon_min, lon_max, lat_min, lat_max]
    return lat_start, lat_end, lon_start, lon_end, extent, latlon_ratio

lat_start, lat_end, lon_start, lon_end, extent, latlon_ratio = latlon_extent(100,160,5,45)  # ✅ 저장할 위경도 범위

year = ['2022']
month = ['08']
day = ['27']
times = ['00']
ens_list = range(0,100)
perturbation_scale_list =[0.2]
factor_list_list = [['z']]

surface_factor = ['MSLP', 'U10', 'V10', 'T2M']
surface_dict = {'MSLP':0, 'U10':1, 'V10':2, 'T2M':3}
upper_factor = ['z', 'q', 't', 'u', 'v']
upper_dict = {'z':0, 'q':1, 't':2, 'u':3, 'v':4}


# ===============================
# ONNX Sessions
# ===============================
options = ort.SessionOptions()
options.enable_cpu_mem_arena= True
options.enable_mem_pattern = False
options.enable_mem_reuse = False

cuda_provider_options_gpu0 = {'arena_extend_strategy': 'kSameAsRequested', 'device_id': 0}
ort_session_6 = ort.InferenceSession(
    rf'{pangu_dir}/pangu_weather_6.onnx', 
    sess_options=options, 
    providers=[('CUDAExecutionProvider', cuda_provider_options_gpu0)]
)
ort_session_24 = ort.InferenceSession(
    rf'{pangu_dir}/pangu_weather_24.onnx', 
    sess_options=options, 
    providers=[('CUDAExecutionProvider', cuda_provider_options_gpu0)]
)

# ===============================
# MAIN LOOP
# ===============================
start = time.time()

for factor_list in factor_list_list:
    for perturbation_scale in perturbation_scale_list:
        for y, m, d, tm in itertools.product(year, month, day, times):
            time_str = f'{y}/{m}/{d}/{tm}UTC'

            input_data_dir = rf'{pangu_dir}/input_data/{time_str}'
            output_data_dir = rf'/data03/Pangu_TC_ENS/output_data/{time_str}'

            input_upper = np.load(os.path.join(input_data_dir, 'upper.npy')).astype(np.float32)
            input_surface = np.load(os.path.join(input_data_dir, 'surface.npy')).astype(np.float32)

            # ===============================
            # STD DEV 계산
            # ===============================
            if lat_perturb:
                # 위도별 표준편차(lat-dependent)
                std_dev_upper = np.std(input_upper, axis=3, dtype=np.float32) * perturbation_scale  # (C, L, lat)
            else:
                # 전체 영역 기반 표준편차
                std_dev_upper = np.std(input_upper, axis=(2, 3), dtype=np.float32) * perturbation_scale  # (C, L)
            std_dev_surface = np.std(input_surface, axis=(1, 2), dtype=np.float32) * perturbation_scale

            factor_str = "".join([f"_{f}" for f in factor_list])
            lp_suffix = "_lp" if lat_perturb else ""
            ens_root_dir = rf'/data03/Pangu_TC_ENS/output_data/{time_str}/{perturbation_scale}ENS{factor_str}{lp_suffix}'

            # ===============================
            # ENSEMBLE LOOP
            # ===============================
            for ens in ens_list:
                output_data_dir = os.path.join(ens_root_dir, str(ens))
                os.makedirs(os.path.join(output_data_dir, 'upper'), exist_ok=True)
                os.makedirs(os.path.join(output_data_dir, 'surface'), exist_ok=True)
                
                perturbed_upper = input_upper.copy()
                perturbed_surface = input_surface.copy()

                seed_val = hash((ens, tuple(factor_list), perturbation_scale)) % (2**32)
                rng = np.random.default_rng(seed_val)

                # ================
                # Perturbation
                # ================
                if ens != 0:
                    for factor in factor_list:
                        if factor in upper_dict:
                            idx = upper_dict[factor]
                            for j in range(13):
                                if lat_perturb:
                                    # 위도별 표준편차 적용
                                    noise = rng.normal(0, 1, size=input_upper[idx, j].shape).astype(np.float32)
                                    scale = std_dev_upper[idx, j][..., None]  # (lat, 1)
                                    perturbation = noise * scale
                                else:
                                    # 전체 평균 표준편차 적용
                                    perturbation = rng.normal(0, std_dev_upper[idx, j], input_upper[idx, j].shape)
                                perturbed_upper[idx, j] = input_upper[idx, j] + perturbation.astype(np.float32)

                        elif factor in surface_dict:
                            idx = surface_dict[factor]
                            perturbation = rng.normal(0, std_dev_surface[idx], input_surface[idx].shape)
                            perturbed_surface[idx] = input_surface[idx] + perturbation.astype(np.float32)

                # ================
                # Save initial
                # ================
                np.save(os.path.join(output_data_dir, f'upper/0h'), perturbed_upper[:,:,lat_start: lat_end+1, lon_start:lon_end+1])
                np.save(os.path.join(output_data_dir, f'surface/0h'), perturbed_surface[:,lat_start: lat_end+1, lon_start:lon_end+1])

                perturbed_24, perturbed_surface_24 = perturbed_upper, perturbed_surface

                # ================
                # Forecast loop
                # ================
                for i in range(28):
                    start_i = time.time()
                    predict_interval = 6 * (i + 1)

                    if (i + 1) % 4 == 0:
                        output, output_surface = ort_session_24.run(None, {'input': perturbed_24, 'input_surface': perturbed_surface_24})
                        perturbed_24, perturbed_surface_24 = output, output_surface
                    else:
                        output, output_surface = ort_session_6.run(None, {'input': perturbed_upper, 'input_surface': perturbed_surface})

                    np.save(os.path.join(output_data_dir, f'upper/{predict_interval}h'), output[:,:,lat_start: lat_end+1, lon_start:lon_end+1])
                    np.save(os.path.join(output_data_dir, f'surface/{predict_interval}h'), output_surface[:,lat_start: lat_end+1, lon_start:lon_end+1])

                    perturbed_upper, perturbed_surface = output, output_surface
                    end_i = time.time()
                    print(f'{factor_list} {perturbation_scale}_{ens}ENS{lp_suffix} {i+1}번째 반복 +{predict_interval}h {end_i-start_i:.2f}s')

                end = time.time()
                print(f"{factor_list} {perturbation_scale}_{ens}ENS{lp_suffix}: {end-start:.1f}s")


['z'] 0.2_0ENS_lp 1번째 반복 +6h 6.93s
['z'] 0.2_0ENS_lp 2번째 반복 +12h 4.97s
['z'] 0.2_0ENS_lp 3번째 반복 +18h 4.96s
['z'] 0.2_0ENS_lp 4번째 반복 +24h 4.62s
['z'] 0.2_0ENS_lp 5번째 반복 +30h 4.96s
['z'] 0.2_0ENS_lp 6번째 반복 +36h 4.96s
['z'] 0.2_0ENS_lp 7번째 반복 +42h 4.96s
['z'] 0.2_0ENS_lp 8번째 반복 +48h 4.96s
['z'] 0.2_0ENS_lp 9번째 반복 +54h 4.33s
['z'] 0.2_0ENS_lp 10번째 반복 +60h 4.97s
['z'] 0.2_0ENS_lp 11번째 반복 +66h 4.96s
['z'] 0.2_0ENS_lp 12번째 반복 +72h 4.96s
['z'] 0.2_0ENS_lp 13번째 반복 +78h 4.96s
['z'] 0.2_0ENS_lp 14번째 반복 +84h 4.31s
['z'] 0.2_0ENS_lp 15번째 반복 +90h 4.96s
['z'] 0.2_0ENS_lp 16번째 반복 +96h 4.96s
['z'] 0.2_0ENS_lp 17번째 반복 +102h 4.96s
['z'] 0.2_0ENS_lp 18번째 반복 +108h 4.96s
['z'] 0.2_0ENS_lp 19번째 반복 +114h 4.32s
['z'] 0.2_0ENS_lp 20번째 반복 +120h 4.96s
['z'] 0.2_0ENS_lp 21번째 반복 +126h 4.96s
['z'] 0.2_0ENS_lp 22번째 반복 +132h 4.96s
['z'] 0.2_0ENS_lp 23번째 반복 +138h 4.95s
['z'] 0.2_0ENS_lp 24번째 반복 +144h 4.32s
['z'] 0.2_0ENS_lp 25번째 반복 +150h 4.96s
['z'] 0.2_0ENS_lp 26번째 반복 +156h 4.96s
['z'] 0.2_0ENS_lp 27번째 반복 +162h 4.96s


In [ ]:

#%%
import torch
import onnxruntime as ort
import os
import numpy as np
import onnx
import time
import itertools

lat_indices = np.linspace(90, -90, 721)
lon_indices = np.linspace(-180, 180, 1441)[:-1]

def latlon_extent(lon_min, lon_max, lat_min, lat_max):    
    lon_min, lon_max = lon_min-180, lon_max-180  
     
    # 위경도 범위를 데이터의 행과 열 인덱스로 변환
    lat_start = np.argmin(np.abs(lat_indices - lat_max)) 
    lat_end = np.argmin(np.abs(lat_indices - lat_min))
    lon_start = np.argmin(np.abs(lon_indices - lon_min))
    lon_end = np.argmin(np.abs(lon_indices - lon_max))
    latlon_ratio = (lon_max-lon_min)/(lat_max-lat_min)
    extent=[lon_min, lon_max, lat_min, lat_max]
    return lat_start, lat_end, lon_start, lon_end, extent, latlon_ratio

lat_start, lat_end, lon_start, lon_end, extent, latlon_ratio = latlon_extent(250,310,5,45)  

# Maximum value at: 28.25, 151.0 with value: 31.951063931819963
# Minimum value at: 22.0, 152.25 with value: -22.832135723733153
max_lat = 28.25
max_lon = 151.0
min_lat = 22.0
min_lon = 152.25
max_lat_idx, max_lon_idx = np.argmin(np.abs(lat_indices - 28.25)), np.argmin(np.abs(lon_indices - 151.0))
min_lat_idx, min_lon_idx = np.argmin(np.abs(lat_indices - 22.0)), np.argmin(np.abs(lon_indices - 152.25))


year = ['2022']
month = ['08']
day = ['27']
times = ['00']
# ens_num = 100
ens_list = range('pos', 'neg')
factor_list_list = [['z']] 
# surface_factors.sort()
# upper_factors.sort()
# surface_str = "".join([f"_{factor}" for factor in surface_factors])  # 각 요소 앞에 _ 추가
# upper_str = "".join([f"_{factor}" for factor in upper_factors])  # 각 요소 앞에 _ 추가
pangu_dir = r'/home1/jek/Pangu-Weather'

surface_factor = ['MSLP', 'U10', 'V10', 'T2M']
surface_dict = {'MSLP':0, 'U10':1, 'V10':2, 'T2M':3}
upper_factor = ['z', 'q', 't', 'u', 'v']
upper_dict = {'z':0, 'q':1, 't':2, 'u':3, 'v':4}


# Set the behavior of onnxruntime
options = ort.SessionOptions()
options.enable_cpu_mem_arena= True
options.enable_mem_pattern = False
options.enable_mem_reuse = False

# Increase the number for faster inference and more memory consumption
# options.intra_op_num_threads = 1

# Set the behavior of cuda provider for the first GPU
cuda_provider_options_gpu0 = {'arena_extend_strategy': 'kSameAsRequested', 'device_id': 1}

# Set the behavior of cuda provider for the second GPU
cuda_provider_options_gpu1 = {'arena_extend_strategy': 'kSameAsRequested', 'device_id': 1}

# Initialize onnxruntime session for Pangu-Weather Models on different GPUs
ort_session_6 = ort.InferenceSession(rf'{pangu_dir}/pangu_weather_6.onnx', sess_options=options, providers=[('CUDAExecutionProvider', cuda_provider_options_gpu0)])
ort_session_24 = ort.InferenceSession(rf'{pangu_dir}/pangu_weather_24.onnx', sess_options=options, providers=[('CUDAExecutionProvider', cuda_provider_options_gpu0)])


start = time.time()

for factor_list in factor_list_list:
    for perturbation_scale in perturbation_scale_list:
        for y, m, d, tm in itertools.product(year, month, day, times):
            time_str = f'{y}/{m}/{d}/{tm}UTC'

            input_data_dir = rf'{pangu_dir}/input_data/{time_str}'
            output_data_dir = rf'/data03/Pangu_TC_ENS/output_data/{time_str}'

            input_upper = np.load(os.path.join(input_data_dir, 'upper.npy')).astype(np.float32)
            input_surface = np.load(os.path.join(input_data_dir, 'surface.npy')).astype(np.float32)


            
            std_dev_upper = np.std(input_upper, axis=(2, 3), dtype=np.float32)*perturbation_scale
            std_dev_surface = np.std(input_surface, axis=(1, 2), dtype=np.float32)*perturbation_scale


            factor_str = "".join([f"_{f}" for f in factor_list])

            for ens in ens_list:
                output_data_dir = rf'/data03/Pangu_TC_ENS/output_data/{time_str}/{perturbation_scale}ENS{factor_str}/{ens}'
                # output_data_dir = rf'/data03/Pangu_TC_ENS/output_data/{time_str}/{ens}'
                
                if not os.path.exists(os.path.join(output_data_dir, f'upper')):
                    os.makedirs(os.path.join(output_data_dir, f'upper'))
                if not os.path.exists(os.path.join(output_data_dir, f'surface')):
                    os.makedirs(os.path.join(output_data_dir, f'surface'))
                
                
                perturbed_upper = input_upper.copy()
                perturbed_surface = input_surface.copy()
                seed_val = hash((ens, tuple(factor_list), perturbation_scale)) % (2**32)
                rng = np.random.default_rng(seed_val)  # 새로운 난수 생성기
                # Perturbation 생성 및 적용     
                if ens == 0:
                    pass
                    

                    
                np.save(os.path.join(output_data_dir, f'upper/0h'), perturbed_upper[:,:,lat_start: lat_end+1, lon_start:lon_end+1])
                np.save(os.path.join(output_data_dir, f'surface/0h'), perturbed_surface[:,lat_start: lat_end+1, lon_start:lon_end+1])

                perturbed_24, perturbed_surface_24 = perturbed_upper, perturbed_surface

                for i in range(28):
                    start_i = time.time()
                    predict_interval = 6*(i+1)
                    if (i+1) % 4 == 0:
                        output, output_surface = ort_session_24.run(None, {'input':perturbed_24, 'input_surface':perturbed_surface_24})
                        perturbed_24, perturbed_surface_24 = output, output_surface
                        np.save(os.path.join(output_data_dir, f'upper/{predict_interval}h'), output[:,:,lat_start: lat_end+1, lon_start:lon_end+1])
                        np.save(os.path.join(output_data_dir, f'surface/{predict_interval}h'), output_surface[:,lat_start: lat_end+1, lon_start:lon_end+1])
                        


                    # 6시간 간격도 저장하고 싶으면 주석 해제
                    else:
                        output, output_surface = ort_session_6.run(None, {'input':perturbed_upper, 'input_surface':perturbed_surface})
                        if predict_interval == 12:
                            q_std = np.std( output[upper_dict['q']], axis=(1,2))
                            output[upper_dict['q']][:,min_lat_idx-6:min_lat_idx+7,min_lon_idx-6:min_lon_idx+7] -= q_std[:, None, None]
                            output[upper_dict['q']][:,max_lat_idx-6:max_lat_idx+7,max_lon_idx-6:max_lon_idx+7] += q_std[:, None, None]
                        np.save(os.path.join(output_data_dir, f'upper/{predict_interval}h'), output[:,:,lat_start: lat_end+1, lon_start:lon_end+1])
                        np.save(os.path.join(output_data_dir, f'surface/{predict_interval}h'), output_surface[:,lat_start: lat_end+1, lon_start:lon_end+1])

                    perturbed_upper, perturbed_surface = output, output_surface
                    end_i = time.time()
                    print(f'{factor_list} {perturbation_scale}_{ens}ENS {i+1}번째 반복 +{predict_interval}h {end_i-start_i}s')
                

                end = time.time()
                print(f"{factor_list} {perturbation_scale}_{ens}ENS: {end-start}s")

In [ ]:
#!/usr/bin/env python
from ecmwfapi import ECMWFDataServer

server = ECMWFDataServer()

server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27/to/2022-08-27",
    "expver": "prod",
    "grid": "0.5/0.5",
    "levelist": "200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "origin": "ecmf",
    "param": "130/131/132/133",
    "step": "0/6/12/18/24/30/36/42/48/54/60/66/72/78/84/90/96/102/108/114/120/126/132/138/144/150/156/162/168",
    "time": "00:00:00",
    "type": "fc",
    "target": "output"
})

server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27/to/2022-08-27",
    "expver": "prod",
    "grid": "0.5/0.5",
    "levelist": "200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "origin": "rksl",
    "param": "130/131/132",
    "step": "0/6/12/18/24/30/36/42/48/54/60/66/72/78/84/90/96/102/108/114/120/126/132/138/144/150/156/162/168",
    "time": "00:00:00",
    "type": "fc",
    "target": "output"
})

server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27/to/2022-08-27",
    "expver": "prod",
    "grid": "0.5/0.5",
    "levelist": "50/200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "origin": "ecmf",
    "param": "156",
    "step": "0/6/12/18/24/30/36/42/48/54/60/66/72/78/84/90/96/102/108/114/120/126/132/138/144/150/156/162/168",
    "time": "00:00:00",
    "type": "fc",
    "target": "output"
})

server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27/to/2022-08-27",
    "expver": "prod",
    "grid": "0.5/0.5",
    "levelist": "50/200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "origin": "rksl",
    "param": "156",
    "step": "0/6/12/18/24/30/36/42/48/54/60/66/72/78/84/90/96/102/108/114/120/126/132/138/144/150/156/162/168",
    "time": "00:00:00",
    "type": "fc",
    "target": "output"
})

ModuleNotFoundError: No module named 'ecmwfapi'

# TIGGE Ensemble Data recive

In [ ]:
from ecmwfapi import ECMWFDataServer
server = ECMWFDataServer()

# TQUV control forecast
# server.retrieve({
#     "class": "ti",
#     "dataset": "tigge",
#     "date": "2022-08-27",
#     "expver": "prod",
#     "grid": "0.25/0.25",
#     "levelist": "200/250/300/500/700/850/925/1000",
#     "levtype": "pl",
#     "origin": "ecmf",
#     "param": "130/131/132/133",   
#     "step": "0/to/240/by/6",
#     "time": "00:12:00",
#     "type": "cf",
#     "target": "/data03/Pangu_TC_ENS/TIGGE/ENS_data/ecmf_cf_tquv_12h.grib"
# })

# # Z control forecast
# server.retrieve({
#     "class": "ti",
#     "dataset": "tigge",
#     "date": "2022-08-27",
#     "expver": "prod",
#     "grid": "0.25/0.25",
#     "levelist": "50/200/250/300/500/700/850/925/1000",
#     "levtype": "pl",
#     "origin": "ecmf",
#     "param": "156",   
#     "step": "0/to/240/by/6",
#     "time": "00:12:00",
#     "type": "cf",
#     "target": "/data03/Pangu_TC_ENS/TIGGE/ENS_data/ecmf_cf_z_12h.grib"
# })

# TQUV perturbed forecast
server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27",
    "expver": "prod",
    "grid": "0.25/0.25",
    "levelist": "200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "number": "1/to/50",
    "origin": "ecmf",
    "param": "130",
    "step": "0/to/168/by/6",
    "time": "00:00:00",
    "type": "pf",
    "target": "/data03/Pangu_TC_ENS/TIGGE/ENS_data/ecmf_pf_t.grib"
})

server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27",
    "expver": "prod",
    "grid": "0.25/0.25",
    "levelist": "200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "number": "1/to/50",
    "origin": "ecmf",
    "param": "133",
    "step": "0/to/168/by/6",
    "time": "00:00:00",
    "type": "pf",
    "target": "/data03/Pangu_TC_ENS/TIGGE/ENS_data/ecmf_pf_q.grib"
})
server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27",
    "expver": "prod",
    "grid": "0.25/0.25",
    "levelist": "200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "number": "1/to/50",
    "origin": "ecmf",
    "param": "131",
    "step": "0/to/168/by/6",
    "time": "00:00:00",
    "type": "pf",
    "target": "/data03/Pangu_TC_ENS/TIGGE/ENS_data/ecmf_pf_u.grib"
})
server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27",
    "expver": "prod",
    "grid": "0.25/0.25",
    "levelist": "200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "number": "1/to/50",
    "origin": "ecmf",
    "param": "132",
    "step": "0/to/168/by/6",
    "time": "00:00:00",
    "type": "pf",
    "target": "/data03/Pangu_TC_ENS/TIGGE/ENS_data/ecmf_pf_v.grib"
})

# Z perturbed forecast
server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27",
    "expver": "prod",
    "grid": "0.25/0.25",
    "levelist": "50/200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "number": "1/to/50",
    "origin": "ecmf",
    "param": "156",
    "step": "0/to/168/by/6",
    "time": "00:00:00",
    "type": "pf",
    "target": "/data03/Pangu_TC_ENS/TIGGE/ENS_data/ecmf_pf_z.grib"
})

2025-10-03 11:16:07 ECMWF API python library 1.6.5
2025-10-03 11:16:07 ECMWF API at https://api.ecmwf.int/v1
2025-10-03 11:16:08 Welcome kim jaeeon
2025-10-03 11:16:11 In case of problems, please check https://confluence.ecmwf.int/display/WEBAPI/Web+API+FAQ or contact servicedesk@ecmwf.int
2025-10-03 11:16:11 ------------ WARNING ------------
2025-10-03 11:16:11 Access to this dataset is transitioning to a new interface, dates to be announced soon
2025-10-03 11:16:11 For more information on how to access this data in the future, visit https://confluence.ecmwf.int/x/-wUiEw
2025-10-03 11:16:11 ---------------------------------
2025-10-03 11:16:12 Request submitted
2025-10-03 11:16:12 Request id: 68df3155714fcaefa95d8aab
2025-10-03 11:16:12 Request is submitted
2025-10-03 11:16:14 Calling 'nice mars /tmp/20251003-0210/36/tmp-_mars-KU9NoF-b1429b229e7462c6f78b5b40702ca6b2.req'
2025-10-03 11:16:14 Forcing MIR_CACHE_PATH=/data/ec_coeff
2025-10-03 11:16:14 mars - WARN -
2025-10-03 11:16:14 mar

In [ ]:
# TQUV control forecast
server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27",
    "expver": "prod",
    "grid": "0.25/0.25",
    "levelist": "200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "origin": "ecmf",
    "param": "130/131/132/133",   
    "step": "0/to/240/by/6",
    "time": "00:12:00",
    "type": "cf",
    "target": "/data03/Pangu_TC_ENS/TIGGE/ENS_data/ecmf_cf_tquv_12h.grib"
})

# Z control forecast
server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2022-08-27",
    "expver": "prod",
    "grid": "0.25/0.25",
    "levelist": "50/200/250/300/500/700/850/925/1000",
    "levtype": "pl",
    "origin": "ecmf",
    "param": "156",   
    "step": "0/to/240/by/6",
    "time": "00:12:00",
    "type": "cf",
    "target": "/data03/Pangu_TC_ENS/TIGGE/ENS_data/ecmf_cf_z_12h.grib"
})

In [7]:
from ecmwfapi import ECMWFDataServer

server = ECMWFDataServer()

for origin, param, suffix in [
    ("ecmf", "130/131/132/133", "params"),
    ("rksl", "130/131/132", "params"),
    ("ecmf", "156", "vort"),
    ("rksl", "156", "vort")
]:
    server.retrieve({
        "class": "ti",
        "dataset": "tigge",
        "date": "2022-08-27/to/2022-08-27",
        "expver": "prod",
        "grid": "0.5/0.5",
        "levelist": "200/250/300/500/700/850/925/1000",
        "levtype": "pl",
        "origin": origin,
        "param": param,
        "step": "0/6/12/18/24/30/36/42/48/54/60/66/72/78/84/90/96/102/108/114/120/126/132/138/144/150/156/162/168",
        "time": "00:00:00",
        "type": "fc",
        "target": f"/data03/Pangu_TC_ENS/TIGGE/ENS_data/tigge_{origin}_{suffix}.grib"
    })


2025-10-01 19:42:06 ECMWF API python library 1.6.5
2025-10-01 19:42:06 ECMWF API at https://api.ecmwf.int/v1
2025-10-01 19:42:07 Welcome kim jaeeon
2025-10-01 19:42:13 In case of problems, please check https://confluence.ecmwf.int/display/WEBAPI/Web+API+FAQ or contact servicedesk@ecmwf.int
2025-10-01 19:42:13 ------------ WARNING ------------
2025-10-01 19:42:13 Access to this dataset is transitioning to a new interface, dates to be announced soon
2025-10-01 19:42:13 For more information on how to access this data in the future, visit https://confluence.ecmwf.int/x/-wUiEw
2025-10-01 19:42:13 ---------------------------------
2025-10-01 19:42:14 Request submitted
2025-10-01 19:42:14 Request id: 68dd04ef07fc9fe5d2678251
2025-10-01 19:42:14 Request is submitted
2025-10-01 19:42:16 Request is queued
2025-10-01 19:44:05 Calling 'nice mars /tmp/20251001-1040/92/tmp-_mars-ExyxYJ-4ed1ae1fe8529e8a6e2928d97b49fb31.req'
2025-10-01 19:44:05 Forcing MIR_CACHE_PATH=/data/ec_coeff
2025-10-01 19:44:05

In [11]:
import xarray as xr

for grib_file in [
    "tigge_ecmf_params.grib",
    "tigge_rksl_params.grib",
    "tigge_ecmf_vort.grib",
    "tigge_rksl_vort.grib"


]:
    
    # import pygrib

    # grbs = pygrib.open("/data03/Pangu_TC_ENS/TIGGE/ENS_data/"+grib_file)
    # print(grbs)
    ds = xr.open_dataset("/data03/Pangu_TC_ENS/TIGGE/ENS_data/"+grib_file, engine="cfgrib")
    print(ds)
    # nc_file = grib_file.replace(".grib", ".nc")
    # ds.to_netcdf(nc_file)
    # print(f"✅ Converted {grib_file} → {nc_file}")

ValueError: unrecognized engine cfgrib must be one of: ['netcdf4', 'scipy', 'gini', 'store', 'zarr']

In [14]:
ds = xr.load_dataset("/data03/Pangu_TC_ENS/TIGGE/ENS_data/tigge_ecmf_params.nc")
# tigge_ecmf_params.nc, tigge_ecmf_vort.nc, tigge_rksl_params.nc, tigge_rksl_vort.nc
ds

<xarray.Dataset> Size: 2GB
Dimensions:    (longitude: 720, latitude: 361, level: 8, time: 29)
Coordinates:
  * longitude  (longitude) float32 3kB 0.0 0.5 1.0 1.5 ... 358.5 359.0 359.5
  * latitude   (latitude) float32 1kB 90.0 89.5 89.0 88.5 ... -89.0 -89.5 -90.0
  * level      (level) int32 32B 200 250 300 500 700 850 925 1000
  * time       (time) datetime64[ns] 232B 2022-08-27 ... 2022-09-03
Data variables:
    t          (time, level, latitude, longitude) float64 482MB 226.0 ... 241.0
    u          (time, level, latitude, longitude) float64 482MB 10.79 ... -3.876
    v          (time, level, latitude, longitude) float64 482MB 2.644 ... 3.093
    q          (time, level, latitude, longitude) float64 482MB 4.363e-06 ......
Attributes:
    Conventions:  CF-1.6
    history:      2025-10-02 07:23:10 GMT by grib_to_netcdf-2.26.0: grib_to_n...

In [15]:
ds.time

<xarray.DataArray 'time' (time: 29)> Size: 232B
array(['2022-08-27T00:00:00.000000000', '2022-08-27T06:00:00.000000000',
       '2022-08-27T12:00:00.000000000', '2022-08-27T18:00:00.000000000',
       '2022-08-28T00:00:00.000000000', '2022-08-28T06:00:00.000000000',
       '2022-08-28T12:00:00.000000000', '2022-08-28T18:00:00.000000000',
       '2022-08-29T00:00:00.000000000', '2022-08-29T06:00:00.000000000',
       '2022-08-29T12:00:00.000000000', '2022-08-29T18:00:00.000000000',
       '2022-08-30T00:00:00.000000000', '2022-08-30T06:00:00.000000000',
       '2022-08-30T12:00:00.000000000', '2022-08-30T18:00:00.000000000',
       '2022-08-31T00:00:00.000000000', '2022-08-31T06:00:00.000000000',
       '2022-08-31T12:00:00.000000000', '2022-08-31T18:00:00.000000000',
       '2022-09-01T00:00:00.000000000', '2022-09-01T06:00:00.000000000',
       '2022-09-01T12:00:00.000000000', '2022-09-01T18:00:00.000000000',
       '2022-09-02T00:00:00.000000000', '2022-09-02T06:00:00.000000000',
       '2022-09-02T12:00:00.000000000', '2022-09-02T18:00:00.000000000',
       '2022-09-03T00:00:00.000000000'], dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 232B 2022-08-27 ... 2022-09-03
Attributes:
    long_name:  time